In [1]:
import mimetypes
import os
import shutil
import magic

CONTAINER_SPECS = [
    {
        "marker": b"IHDR",
        "offset_range": (8, 32),
        "mime": "image/png",
        "ext": ".png",
        "patch": (0, b"\x89PNG\r\n\x1a\n"),
        "issue": "Tampered PNG header",
    },
    {
        "marker": b"WEBP",
        "offset_range": (8, 12),
        "mime": "image/webp",
        "ext": ".webp",
        "patch": None,
        "issue": "Valid WebP container",
    },
    {
        "marker": b"WAVE",
        "offset_range": (8, 12),
        "mime": "audio/wav",
        "ext": ".wav",
        "patch": None,
        "issue": "WAV audio container",
    },
    {
        "marker": b"AVI ",
        "offset_range": (8, 12),
        "mime": "video/x-msvideo",
        "ext": ".avi",
        "patch": None,
        "issue": "AVI video container",
    },
]

KNOWN_EXTS = {
    "text/plain": {".txt", ".log", ".cfg", ".conf", ".ini", ".url", ".srt", ""},
    "application/x-executable": {"", ".bin"},
    "application/x-pie-executable": {"", ".bin"},
    "application/x-sharedlib": {".so", ""},
    "application/json": {".json", ".parts.json"},
    "video/x-matroska": {".mkv"},
    "video/mp4": {".mp4", ".mov"},
    "audio/x-wav": {".wav"},
    "image/webp": {".webp"},
    "image/png": {".png"},
    "image/jpeg": {".jpg", ".jpeg"},
}


def probe_file(filepath):
    if not os.path.isfile(filepath) or os.path.getsize(filepath) == 0:
        return None

    size = os.path.getsize(filepath)
    ext = os.path.splitext(filepath)[1].lower()

    with open(filepath, "rb") as f:
        head = f.read(min(2048, size))

    # Handle RIFF ambiguity explicitly (WebP vs WAV vs AVI)
    if head.startswith(b"RIFF") and len(head) >= 12:
        tag = head[8:12]
        if tag == b"WEBP":
            return {
                "ext": ".webp",
                "mime": "image/webp",
                "issue": "WebP image",
                "patch": None,
            }
        elif tag == b"WAVE":
            return {
                "ext": ".wav",
                "mime": "audio/wav",
                "issue": "WAV audio",
                "patch": None,
            }
        elif tag == b"AVI ":
            return {
                "ext": ".avi",
                "mime": "video/x-msvideo",
                "issue": "AVI video",
                "patch": None,
            }

    # Handle broken PNGs where bytes 0-3 were tampered but IHDR is intact
    if b"IHDR" in head[:32] and not head.startswith(b"\x89PNG\r\n\x1a\n"):
        return {
            "ext": ".png",
            "mime": "image/png",
            "issue": "Tampered PNG header",
            "patch": (0, b"\x89PNG\r\n\x1a\n"),
        }

    # Fall back to libmagic for standard containers
    try:
        raw_mime = magic.Magic(mime=True).from_file(filepath)
    except Exception:
        raw_mime = "application/octet-stream"

    if raw_mime not in ("application/octet-stream", "data"):
        matched_exts = set(mimetypes.guess_all_extensions(raw_mime)).union(
            KNOWN_EXTS.get(raw_mime, set())
        )
        target_ext = mimetypes.guess_extension(raw_mime) or ""

        if raw_mime == "video/x-matroska":
            target_ext = ".mkv"

        if matched_exts and ext not in matched_exts:
            return {
                "ext": target_ext,
                "mime": raw_mime,
                "issue": f"Extension mismatch ({raw_mime})",
                "patch": None,
            }
        return None

    return None

In [2]:
TARGET_DIR = "test_files"

collected = []
for root, _, files in os.walk(TARGET_DIR):
    for f in sorted(files):
        if "_recovered" not in f:
            collected.append(os.path.join(root, f))

header = f"{'FILENAME':<35} {'ORIG EXT':<10} {'TRUE EXT':<10} {'RECOVERED FILE'}"
print(header)
print("-" * len(header))

recovered_count = 0
for path in collected:
    result = probe_file(path)
    if not result:
        continue

    orig_ext = os.path.splitext(path)[1] or "(none)"
    target_ext = result["ext"] if result["ext"] else ".bin"

    base, _ = os.path.splitext(path)
    out_path = f"{base}_recovered{target_ext}"

    # Generate isolated recovered duplicate
    shutil.copy2(path, out_path)
    if result["patch"]:
        offset, patch_bytes = result["patch"]
        with open(out_path, "r+b") as f:
            f.seek(offset)
            f.write(patch_bytes)

    recovered_count += 1
    fname = os.path.basename(path)
    if len(fname) > 33:
        fname = fname[:30] + "..."
    print(
        f"{fname:<35} {orig_ext:<10} {target_ext:<10} {os.path.basename(out_path)}"
    )

print("-" * len(header))
print(f"Total recovered files generated: {recovered_count}")

FILENAME                            ORIG EXT   TRUE EXT   RECOVERED FILE
------------------------------------------------------------------------
IMG_3041.webp                       .webp      .txt       IMG_3041_recovered.txt
[Erai-raws] Shingeki no Kyojin...   .part01    .txt       [Erai-raws] Shingeki no Kyojin - 01 [720p][Dual Audio].mkv_recovered.txt
[Erai-raws] Shingeki no Kyojin...   .part02    .txt       [Erai-raws] Shingeki no Kyojin - 01 [720p][Dual Audio].mkv_recovered.txt
[Erai-raws] Shingeki no Kyojin...   .part03    .txt       [Erai-raws] Shingeki no Kyojin - 01 [720p][Dual Audio].mkv_recovered.txt
[Erai-raws] Shingeki no Kyojin...   .part04    .txt       [Erai-raws] Shingeki no Kyojin - 01 [720p][Dual Audio].mkv_recovered.txt
blank                               (none)     .png       blank_recovered.png
dj_cara_after_hours.wav             .wav       .txt       dj_cara_after_hours_recovered.txt
polish_cow.webp                     .webp      .txt       polish_cow_recovered.